Stage 1: Generating the calibrated data from the MeerKLASS pilot pilot survey
Author: Brandon Engelbrecht

NOTE:
    For T_el we had to use JY code for it

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Modules Required to import 

import numpy as np
import pickle
import sys
from os import path
import os
from tqdm.notebook import tqdm
import numpy.ma as ma

# Personal file
from data_reduction import *

In [3]:
"""
NOTE:
File names - select the filename for which you would like to run; 
file names have unique paths
"""

# 1551037708   done
# 1551055211   done
# 1553966342   done
# 1554156377   done
# 1556052116   done
# 1556138397   done
# 1562857793   issue

fname = '1551055211'   

if fname == '1551037708' or fname == '1551055211' or fname == '1553966342' or fname=='1554156377':
    folder = 'SCI-20180330-MS-01'
    
if fname == '1555775533' or fname =='1555793534' or fname =='1555861810' or fname =='1556034219' or fname =='1556052116' or fname =='1556120503' or fname =='1556138397' or fname =='1555879611' or fname =='1561650779':
    folder = 'SCI-20190418-MS-01'

if fname == '1562857793':
    folder = ''
    
# ------------------------------------------
# Path to the data files 
data_file_path = '..//..//test1/'
# ------------------------------------------

# Applying the work done in the data_reduction.py file
dat1 = Data_reduction(file_name = fname, folder_name = folder, folder_output=data_file_path)

/anaconda2/lib/python2.7/site-packages/katsdptelstate/telescope_state.py:338: FutureWarning: The telescope state contains pickled values. This is a security risk, but is allowed because MeerKAT data up to March 2019 uses it. You can suppress this warning by setting KATSDPTELSTATE_ALLOW_PICKLE=1 in the environment, or disable pickles by setting KATSDPTELSTATE_ALLOW_PICKLE=0.
  warnings.warn(PICKLE_WARNING, FutureWarning)


## Calculating Calibrated Data

In [6]:
"""
NOTE:
Checking if the KATDAL information is already available, if not running the process to obtain

**{Does take some time to run}
"""


if path.exists(data_file_path+fname+'_katdal_info.p'):
    print 'Files already exists, data is pulled'
    katdal_info = pickle.load(open(data_file_path+fname+'_katdal_info.p', 'rb'))

    nd_s, nd_s0, freqs, nd_s0_pos, timestamps, nd_s0_coords = [katdal_info[i]
                                                               for i in katdal_info.keys()]

else:
    nd_s0, nd_s0_pos, nd_s, nd_s0_coords = dat1.get_nd_times()

    freqs, timestamps = dat1.freqs, dat1.timestamps

    # Pickling the inforamation from dat1 been pickles
    katdal_info = {"nd_s0":nd_s0, 
                   "nd_s0_pos":nd_s0_pos,
                   "nd_s":nd_s,
                   "nd_s0_coords":nd_s0_coords,
                   "frequency":freqs,
                   "time":timestamps 
    }

    pickle.dump(katdal_info, open(data_file_path+fname+'_katdal_info.p', 'wb'))

Time Position-File exists
Noise Diode file exists
Done
1. Time for when noise diode off; 
2. The position when noise diode is off in the scan; 
3. Scan time; 
4. Az&El when noise diode is off during scan


In [7]:
"""
NOTE:
Calculating the Time-Ordered-Data values for antennae and their respective bandpass in the HH and VV polarisations

**{Process does take a while to run}
"""

# Channel from which we are starting the freqeuncy. This channel is ~ 980 MHz
channel_start_choice = 600

if path.exists(data_file_path+fname+'_calibrated_antennae_data.p'):
    print 'Files already exists, data is pulled'
    cali_data = pickle.load(open(data_file_path+fname+'_calibrated_antennae_data.p', 'rb'))
    
    good_antennae = cali_data['ant']
    temperature_tod = cali_data['cali_tod']
    
else:
    print 'Data does not exist'
    temperature_ = []          # The temperature TOD array
    b_pass_hh = []             # Bandpass in the HH
    b_pass_vv = []             # Bandpass in the VV
    good_antennae = []         # List of good antennae
    
    
    # Will run through the list of antennae from the MeerKAT antenna list
    for i in tqdm(range(len(dat1.obs_data.ants)), unit='ant'):
        try: 
            # Looking the noise diode switched off in the scan portion and looking from channel=600
            TOD = dat1.TOD(ant_no=i, nd_off=nd_s0_pos, c_start=channel_start_choice)
            temperature_.append(TOD[0])
            b_pass_hh.append(TOD[1])
            b_pass_vv.append(TOD[2])

        # Exceptions from missing information
        except IOError:
            print 'Antenna - '+str(dat1.obs_data.ants[i])[0:4]+' data missing'
            continue
            
        except ValueError:
            print 'Antenna - '+str(dat1.obs_data.ants[i])[0:4]+' zero-size array to reduction operation maximum which has no identity'
            continue
        else:
            good_antennae.append(str(dat1.obs_data.ants[i])[0:4])
    
    
    frequency_end_pos = TOD[3]                 # The channel position or the index ositopn of the last frequency
    temperature_tod = np.array(temperature_)   # The lsit of TOD data now in array form
    
    
    # Dictionary of the data for the calibration section
    cali_tod = {
        'ant': np.array(good_antennae),       # List of good antennae
        'cali_tod': temperature_tod,          # Calibrated TOD datat
        'freq_end_pos': frequency_end_pos     # Frequency postion or channel position
    }

    # Pickling the dictionary for output
    pickle.dump(cali_tod, open(data_file_path+fname+'_calibrated_antennae_data.p', 'wb'))

# Resetting the dictionary for memory purposes
cali_tod = {}

Data does not exist


Antenna - m025 data missing
Antenna - m041 data missing


data_reduction.py:173: RuntimeWarning: divide by zero encountered in divide
  


Antenna - m042 zero-size array to reduction operation maximum which has no identity



In [8]:
# Loading the calibrated data
cali_data = pickle.load(open(data_file_path+fname+'_calibrated_antennae_data.p', 'rb'))


# Getting the keys from the data 
good_antennae = np.array(cali_data['ant'])
temperature_tod = cali_data['cali_tod']
channel_end_position = cali_data['freq_end_pos']

## Calculating Background Model (Noise Model)

In [9]:
# The frequency band for the MeerKLASS project
freq_band = freqs[channel_start_choice : channel_start_choice+frequency_end_pos]

In [10]:
"""
Location of the various data, done y Dr Jingying Wang

**{Does take a while to run}
"""
loc = '/idia/projects/hi_im/raw_vis/katcali_output/'

t_rec_h, t_rec_v = [], []
t_gal_h, t_gal_v = [], []
t_map_h, t_map_v = [], []

d4_mask_list = [] 
# Looping through all antennae for the different models
for ant_no, i in enumerate(tqdm(good_antennae, unit='ant')):
    # checking to see if the masks exist or not and re-writting the code
    try:
        # Lvl 3 mask
        d3_h = pickle.load(open(loc+'level3_output/'+str(fname)+'_'+i+'h_level3_data'))
        d3_v = pickle.load(open(loc+'level3_output/'+str(fname)+'_'+i+'v_level3_data'))

    except IOError:
        print 'Antenna - '+i+' Lvl-3 mask missing'
        good_antennae = np.delete(good_antennae, ant_no)
        temperature_tod = np.delete(arr=temperature_tod, obj=ant_no, axis=0)
        continue
        
    try:
        # Lvl 4 mask
        d4_mask = pickle.load(open(loc+'level4_output/mask/'+str(fname)+'_'+i+'_level4_mask'))
        d4_mask_list.append(d4_mask['Inten_mask'][nd_s0_pos, 600:frequency_end_pos+600].T)
        
    except IOError:
        print 'Antenna - '+i+' Lvl-4 mask missing'
        good_antennae = np.delete(good_antennae, ant_no)
        temperature_tod = np.delete(arr=temperature_tod, obj=ant_no, axis=0)
        continue

    # T_rec information
    t_rec_h.append(d3_h['Tsm_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    t_rec_v.append(d3_v['Tsm_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    
    # T_gal information
    t_gal_h.append(d3_h['Tgal_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    t_gal_v.append(d3_v['Tgal_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    
    # T_map information
    t_map_h.append(d3_h['T_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    t_map_v.append(d3_v['T_map'][nd_s0_pos, 600:frequency_end_pos+600].T)
    
t_rec_h = np.ma.array(t_rec_h)
t_rec_v = np.ma.array(t_rec_v)

t_gal_h = np.ma.array(t_gal_h)
t_gal_v = np.ma.array(t_gal_v)

t_map_h = np.ma.array(t_map_h)
t_map_v = np.ma.array(t_map_v)

d4_mask = np.array(d4_mask_list)

In [11]:
# Updatting the list
cali_data['ant'] = good_antennae
cali_data['cali_tod'] = temperature_tod

In [12]:
# Checking the data and saving it

if os.path.exists(data_file_path+fname+'_calibrated_antennae_data.p'):
    os.remove(data_file_path+fname+'_calibrated_antennae_data.p')
    pickle.dump(cali_data, open(data_file_path+fname+'_calibrated_antennae_data.p', 'wb'))

In [13]:
# Resetting the variales
temperature_tod = 0
cali_data = 0

#### T-receiver

In [14]:

# Getting the mean of all antennae
t_rec_h_avg = np.ma.median(t_rec_h, axis=0)
t_rec_v_avg = np.ma.median(t_rec_v, axis=0)

# Averaging across time for the mean antennae
t_rec_h_across_t = np.ma.median(t_rec_h_avg, axis=0)
t_rec_v_across_t = np.ma.median(t_rec_v_avg, axis=0)

# Getting the mask for level 4 for the different antennae
t_rec_h_l4, t_rec_v_l4 = [], []
for i in range(t_rec_h.shape[0]):
    t_rec_h_l4.append(np.ma.masked_array(t_rec_h[i], mask=d4_mask[i]))
    t_rec_v_l4.append(np.ma.masked_array(t_rec_v[i], mask=d4_mask[i]))

t_rec_h_l4 = np.ma.array(t_rec_h_l4)
t_rec_v_l4 = np.ma.array(t_rec_v_l4 )

# Getting the median of all antennae
t_rec_h_avg_l4 = np.ma.median(t_rec_h_l4, axis=0)
t_rec_v_avg_l4 = np.ma.median(t_rec_v_l4, axis=0)

# Getting the frequency list for RBF interpolation
t_rec_h_avg_l4_freq = np.ma.median(t_rec_h_avg_l4, axis=1)
t_rec_v_avg_l4_freq = np.ma.median(t_rec_v_avg_l4, axis=1)

'''Change up, switching from using the first timestamp to using the average timestamp'''
freq_trec_h_list = np.where(t_rec_h_avg_l4_freq.mask==False)[0]
freq_trec_v_list = np.where(t_rec_v_avg_l4_freq.mask==False)[0]


In [15]:
t_rec_0 = [np.mean(t_rec_h[0], axis=1), np.mean(t_rec_v[0], axis=1)]

In [16]:
#Reset the memory
t_rec_h = 0
t_rec_v = 0

In [17]:
# Using RBF-linear as it seems to be the best
'''Got to check this one---Reason for linear can be found in JNote version 1 calculations'''
rbf_lin_h = Rbf(freq_band[freq_trec_h_list], 
                t_rec_h_avg_l4_freq[freq_trec_h_list], function='linear')

rbf_lin_v = Rbf(freq_band[freq_trec_v_list], 
                t_rec_v_avg_l4_freq[freq_trec_v_list], function='linear')

In [18]:
# Interpolating over the full frequency band
t_rec_h_f_interp = rbf_lin_h(freq_band)[:, None]
t_rec_v_f_interp = rbf_lin_v(freq_band)[:, None]

# Setting up the time and normalizing it
t_rec_h_across_t_norm = t_rec_h_across_t[None, :] / np.max(t_rec_h_across_t)
t_rec_v_across_t_norm = t_rec_v_across_t[None, :] / np.max(t_rec_v_across_t)

# New TOD interpolated
t_rec_h_TOD = t_rec_h_f_interp * t_rec_h_across_t_norm
t_rec_v_TOD = t_rec_v_f_interp * t_rec_v_across_t_norm

# Combining polarisations together
t_rec_TOD = (t_rec_h_TOD+t_rec_v_TOD)/2

#### T-galactic

In [19]:
t_gal_h_avg = np.ma.median(t_gal_h, axis=0)
t_gal_v_avg = np.ma.median(t_gal_v, axis=0)

t_gal_TOD = (t_gal_h_avg + t_gal_v_avg) / 2

t_gal_h = 0
t_gal_v = 0

.

#### T-elevation

(A speacil case for now.)

In [ ]:
# We have a script in this location that runs for the blocks T_el   (This script needs to be added here as well)
tel_loc = '/users/bengelbrecht/PhD_Work/Satellite_Code/My_files_2/Work_in_progress/T_elevation_jobs/'

t_el_h, t_el_v = [], []
for i in tqdm(good_antennae): # Error missing m031
# for i in tqdm(range(60)):
    d_h = pickle.load(open(tel_loc+fname+'_'+i+'h_full_t_el', 'rb'))
    d_v = pickle.load(open(tel_loc+fname+'_'+i+'v_full_t_el', 'rb'))

    t_el_h.append(d_h['Tel_map'][600:frequency_end_pos+600, nd_s0_pos] * 1000)
    t_el_v.append(d_v['Tel_map'][600:frequency_end_pos+600, nd_s0_pos] * 1000)
    
t_el_h = np.ma.array(t_el_h)
t_el_v = np.ma.array(t_el_v)

t_el_h_avg =  np.ma.median(t_el_h, axis=0)
t_el_v_avg =  np.ma.median(t_el_v, axis=0)

t_el = (t_el_h_avg + t_el_v_avg)/2

t_el_h, t_el_v, t_el_h_avg, t_el_v_avg = 0,0,0,0

In [ ]:
# t_el_0 = [t_el_h[0], t_el_v[0]]
# pickle.dump(t_el_0, open('Data/'+fname+'_m000_t_el.p', 'wb'))

In [ ]:
# Finding spikes in the data
threshold_no = 2

try:
    s_point = np.where(abs(t_el[0,1:] - t_el[0,0:-1]) > threshold_no)[0][0]
    e_point = np.where(abs(t_el[0,1:] - t_el[0,0:-1]) > threshold_no)[0][-1]

    area_of_interest = np.arange(s_point, e_point+1, 1)

    # Making a mask for the area
    masking_area = np.zeros(t_el.shape)
    masking_area[:, area_of_interest] = 1

    # New T_el
    t_le_masked = ma.masked_array(t_el, mask=masking_area)

    # Deleting area of interests
    t_le_masked_across_t = np.delete(t_le_masked[0], area_of_interest)
    nd_s0_2 = np.delete(nd_s0, area_of_interest)

    # RBF  in the temporal 
    rbf_tle_time = Rbf(nd_s0_2, t_le_masked_across_t)

    # The frequency dependance, nomralized
    tle_freq = t_le_masked[:, 0] / np.ma.max(t_le_masked[:, 0])

    # Final T_le
    t_el_final = tle_freq[:, None] * rbf_tle_time(nd_s0)[None, :] / 1000    #[K]
    
except IndexError:
    print 'Data has no spikes'
    
    # Final T_le
    t_el_final = t_el / 1000    #[K]
    


.

In [ ]:
# CMB Temperature
T_cmb = 2.73 #[K]

.

In [ ]:
# Total Noise Temperature

T_noise = (t_rec_TOD + t_gal_TOD + t_el_final + T_cmb)  # [K]

.

In [ ]:
# Getting the average calibrated TOD temeprature  in kelvin
cali_data = pickle.load(open('Data2/'+fname+'_calibrated_antennae_data.p', 'rb'))

temperature_tod_avg = np.mean(cali_data['cali_tod'], 
                              axis=0) / 1000

cali_data = 0

In [ ]:
d = {}
d['T_TOD'] = temperature_tod_avg
d['T_rec'] = t_rec_TOD
d['T_gal'] = t_gal_TOD
d['T_el'] = t_el_final
d['T_noise'] = T_noise

fs=open('Data2/'+fname+'_S1_Noise_contibution.p','wb')
pickle.dump(d,fs,protocol=2)
fs.close()

d = {}

In [ ]:
stop

----------------------------------------------------------

### Plots

In [ ]:
vis_data = dat1.get_vis_data(ant_no=2)[0]
gain_data = dat1.get_gain_data(ant_no=2)[0]

In [ ]:
vis_data_s = vis_data[nd_s0_pos, 
                      600:frequency_end_pos+600]

In [ ]:
tod_data = pickle.load(open('../Step1_Generating_Calibrated_Data/'+fname+'_S1_Noise_contibution.p'))

In [ ]:
tod_data.keys()

In [ ]:
extent_for_2d = [nd_s0[0], nd_s0[-1], freqs[frequency_end_pos+600], freqs[600]]

In [ ]:
plt.figure(figsize=(8,4))
plt.title('Observation no: '+fname, fontdict=font)

hb = plt.imshow(vis_data_s, aspect='auto', 
                extent = extent_for_2d, vmin=None, vmax=None)

plt.xlabel('Time [s]', fontdict=font_s)
plt.ylabel('Frequency [MHz]', fontdict=font_s)
plt.tight_layout()

cbar = plt.colorbar(hb)
cbar.set_label('Raw Visibility Units', rotation=270, fontdict=font_s, 
               labelpad=20, y=0.45)

plt.tight_layout()
plt.savefig('../SARAO_results/'+fname+'raw_vis.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.imshow(temperature_tod_avg.T)
plt.colorbar()

In [ ]:
plt.figure(figsize=(8,4))
plt.title('Observation no: '+fname, fontdict=font)

hb = plt.imshow(temperature_tod_avg.T, aspect='auto', 
                extent = extent_for_2d, vmin=None, vmax=None)

plt.xlabel('Time [s]', fontdict=font_s)
plt.ylabel('Frequency [MHz]', fontdict=font_s)
plt.tight_layout()

cbar = plt.colorbar(hb)
cbar.set_label('Raw Visibility Units', rotation=270, fontdict=font_s, 
               labelpad=20, y=0.45)

plt.tight_layout()
plt.savefig('../SARAO_results/'+fname+'temp.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10,6))


p1 = axs[0,0].imshow((t_rec_h_avg + t_rec_v_avg)/2)
# plt.colorbar(p1,cax=ax1)

axs[0,1].imshow(t_rec_TOD)

axs[1,0].imshow((t_gal_h_avg + t_gal_v_avg)/2)
axs[1,1].imshow(t_gal_TOD)

axs[2,0].imshow((t_el_h_avg + t_el_v_avg)/2)
axs[2,1].imshow(t_el_final)

plt.tight_layout()
plt.show()